In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Reading Dataset

In [ ]:
# filepath to the csv file
filepath = '/content/drive/MyDrive/AI ML/ICT_AI_ML/student_manager/data/OnlineNewsPopularity.csv'

# Reading dataset and assigning the df created to a variable so as to store it
df_news = pd.read_csv(filepath)
df_news.columns = df_news.columns.str.strip()
df_news.head()

# EDA

In [ ]:
df_news.shape

In [ ]:
df_news.info()

In [ ]:
df_news.describe()

In [ ]:
df_news.isnull().sum()

In [ ]:
df_news.duplicated().sum()

In [ ]:
# "url" is an article link and "timedelta" is days since published, both doesn't predict news popularity hence Drop
df_news.drop(columns = ["url", "timedelta"], inplace = True)
df_news.head()

In [ ]:
num_cols = []
cat_cols = []

In [ ]:
num_cols = df_news.select_dtypes(include = ['int','float']).columns
num_cols

In [ ]:
cat_cols = df_news.select_dtypes(include = ['object']).columns
cat_cols    # empty hence no encoding required

### Correlation & Mutual Information

In [ ]:
# correlation gives linear relationships with "shares"
corr_with_target = df_news.corr()['shares'].sort_values(ascending=False)
corr_with_target

In [ ]:
X_temp = df_news.drop(columns = ["shares"])
y_temp = df_news["shares"]

# mutual_info_regression is used because our target column "shares" has numeric/continous values
mi_scores = mutual_info_regression(X_temp, y_temp, random_state = 42)
# for "number to its features mapping" we use pd.Series()
mi_series = pd.Series(mi_scores, index = X_temp.columns).sort_values(ascending = False)
mi_series

### Plots

In [ ]:
# selecting key columns to analyze
key_cols = mi_series.head(10).index.tolist()
key_cols

In [ ]:
# plotting distribution of each key column using histograms
plt.figure(figsize=(14,10))
for i, col in enumerate(key_cols):
    plt.subplot(5, 2, i+1)
    sns.histplot(df_news[col]) # shows spread/skeweness of each variable
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
# plotting boxplots to check for outliers in each column
plt.figure(figsize=(14,10))
for i, col in enumerate(key_cols):
    plt.subplot(5, 2, i+1)
    sns.boxplot(df_news[col].dropna())   # highlights outliers beyond whiskers
    plt.title(col)

plt.tight_layout()
plt.show()

# PreProcessing

## Data cleaning

### Missing Value Handling

In [ ]:
# There is no missing values to be handled.

### Outlier Handling

In [ ]:
out_cols = [col for col in key_cols if col != 'shares']
out_cols

In [ ]:
for col in out_cols:
    Q1 = df_news[col].quantile(0.25)
    Q3 = df_news[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_news[col] = np.where(df_news[col] < lower, lower, df_news[col])
    df_news[col] = np.where(df_news[col] > upper, upper, df_news[col])

### Duplicates Removal

In [ ]:
# There is no duplicates to remove

### Data Split

In [ ]:
# split before scaling to prevent data leakage
X = df_news.drop(columns = ["shares"])
y = df_news["shares"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Data Transformation

### Scaling

In [ ]:
# StandardScaler() uses mean as 0 and std as 1, Data is skewed with outliers and since min-max is sensitive to outliers we use StandardScaler()
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)     # fits only on train data
X_test = scaler.transform(X_test)       # transforms test using train's data

### Encoding

In [ ]:
# No categorical columns after dropping 'url' hence nothing to encode

# Model Building

##SUPPORT VECTOR REGRESSION

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

In [ ]:

# SUPPORT VECTOR REGRESSION (SVR)

# Build model object
svr_model = SVR()

# Train the model
svr_model.fit(X_train, y_train)

# Make predictions
y_pred_svr = svr_model.predict(X_test)

# Evaluate model performance
svr_mae = mean_absolute_error(y_test, y_pred_svr)
svr_mse = mean_squared_error(y_test, y_pred_svr)
svr_rmse = root_mean_squared_error(y_test, y_pred_svr)

print("Mean Absolute Error (SVR) :", svr_mae)
print("Mean Squared Error (SVR) :", svr_mse)
print("Root Mean Squared Error (SVR) :", svr_rmse)

##CROSS VALIDATION

###K-Fold Cross Validation

In [ ]:

# K-Fold Cross Validation
kf_obj = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    SVR(),
    X,
    y,
    cv=kf_obj
)

print("For Support Vector Regressor")
print("Cross Validation Scores :", scores)
print("Mean CV Score :", scores.mean())

###Grid Search

In [ ]:

# Grid Search CV

param_grid_svr = {

    'kernel': ['linear', 'rbf'],

    'C': [0.1, 1, 10, 100],

    'epsilon': [0.01, 0.1, 0.5, 1],

    'gamma': ['scale', 'auto']

}

grid_search_svr = GridSearchCV(

    estimator=SVR(),

    param_grid=param_grid_svr,

    cv=kf_obj,

    scoring='neg_mean_absolute_error',

    n_jobs=-1,

    verbose=1

)

grid_search_svr.fit(X_train, y_train)

print("Best Parameters :", grid_search_svr.best_params_)

print("Best MAE :", -grid_search_svr.best_score_)

In [ ]:
# Evaluate Tuned SVR Model


best_svr_model = grid_search_svr.best_estimator_

y_pred_svr_tuned = best_svr_model.predict(X_test)

svr_mae_tuned = mean_absolute_error(y_test, y_pred_svr_tuned)

svr_mse_tuned = mean_squared_error(y_test, y_pred_svr_tuned)

svr_rmse_tuned = root_mean_squared_error(y_test, y_pred_svr_tuned)

print("Mean Absolute Error (Tuned SVR) :", svr_mae_tuned)

print("Mean Squared Error (Tuned SVR) :", svr_mse_tuned)

print("Root Mean Squared Error (Tuned SVR) :", svr_rmse_tuned)

###Random Search

In [ ]:
# Randomized Search CV

param_dist_svr = {

    'kernel': ['linear', 'rbf'],

    'C': uniform(0.1,100),

    'epsilon': uniform(0.01,1),

    'gamma': ['scale','auto']

}

random_search_svr = RandomizedSearchCV(

    estimator=SVR(),

    param_distributions=param_dist_svr,

    n_iter=30,

    cv=kf_obj,

    scoring='neg_mean_absolute_error',

    random_state=42,

    n_jobs=-1,

    verbose=1

)

random_search_svr.fit(X_train, y_train)

print("Best Parameters :", random_search_svr.best_params_)

print("Best MAE :", -random_search_svr.best_score_)

In [ ]:
# Evaluate Random Search Model

best_random_svr = random_search_svr.best_estimator_

y_pred_random = best_random_svr.predict(X_test)

random_mae = mean_absolute_error(y_test, y_pred_random)

random_mse = mean_squared_error(y_test, y_pred_random)

random_rmse = root_mean_squared_error(y_test, y_pred_random)

print("Mean Absolute Error (Random Search SVR) :", random_mae)

print("Mean Squared Error (Random Search SVR) :", random_mse)

print("Root Mean Squared Error (Random Search SVR) :", random_rmse)